# Experiment 45: Disorder Information Loss Investigation

**Goal:** Identify exactly where and why the codec loses disorder prediction signal (93-95% retention vs 97-100% for all other tasks).

**Pipeline under investigation:**
```
Raw 1024d → [ABTT3: center + remove top-3 PCs] → [RP to 768d] → [PQ M=192] → decoded 768d
```

**5 investigation layers:**
1. Per-protein, per-stage Spearman rho decomposition
2. Stratified protein selection (worst/average/winners)
3. Subspace forensics (which directions carry disorder signal)
4. Residue-level error maps
5. Targeted recovery experiments

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.path.abspath("__file__")) if os.path.exists("45_disorder_helpers.py") else "..")
sys.path.insert(0, ".")
sys.path.insert(0, "experiments")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from pathlib import Path

from importlib.util import spec_from_file_location, module_from_spec
_spec = spec_from_file_location("helpers", "experiments/45_disorder_helpers.py")
h = module_from_spec(_spec)
_spec.loader.exec_module(h)

from src.one_embedding.preprocessing import compute_corpus_stats, center_embeddings
from src.one_embedding.codec_v2 import OneEmbeddingCodec

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
print("Setup complete")

## Layer 1: Per-Stage Rho Decomposition (CheZOD)

In [ ]:
print("Loading CheZOD data...")
embs, scores, train_ids, test_ids = h.load_chezod_data()
print(f"  Train: {len(train_ids)}, Test: {len(test_ids)}")
print(f"  Embedding dim: {next(iter(embs.values())).shape[1]}")

In [ ]:
# Fit corpus stats on training set
train_embs = {k: embs[k] for k in train_ids}
corpus_stats = compute_corpus_stats(train_embs, n_sample=50_000, n_pcs=5, seed=42)
print(f"Mean vec shape: {corpus_stats['mean_vec'].shape}")
print(f"Top PCs shape: {corpus_stats['top_pcs'].shape}")
print(f"Explained variance (top 5): {corpus_stats['explained_variance']}")

# Build embeddings at each pipeline stage
stages = {}
stages["raw"] = embs
print("\nApplying ABTT3...")
stages["abtt"] = h.apply_abtt(embs, corpus_stats, k=3)
print("Applying RP 768d...")
stages["rp768"] = h.apply_rp(stages["abtt"], d_out=768, seed=42)

# For PQ: need a fitted codec
print("Fitting PQ codec...")
codec = OneEmbeddingCodec(d_out=768, quantization="pq", pq_m=192)
codec.fit(train_embs)
stages["pq"] = h.apply_pq(stages["rp768"], codec)

print(f"\nStages built:")
for name, st in stages.items():
    sample = next(iter(st.values()))
    print(f"  {name}: dim={sample.shape[1]}")

In [ ]:
# Train per-stage probes and compute per-protein rho
probes = {}
per_protein_rhos = {}  # {stage: {pid: rho}}
pooled_rhos = {}  # {stage: float}

for stage_name, stage_embs in stages.items():
    print(f"\n{'='*60}")
    print(f"Stage: {stage_name}")
    print(f"{'='*60}")

    probe = h.train_ridge_probe(stage_embs, scores, train_ids)
    probes[stage_name] = probe
    print(f"  Best alpha: {probe.alpha_}")

    rhos = h.per_protein_rho(stage_embs, scores, test_ids, probe)
    per_protein_rhos[stage_name] = rhos
    print(f"  Per-protein rho: mean={np.mean(list(rhos.values())):.4f}, "
          f"median={np.median(list(rhos.values())):.4f}")

    prho = h.pooled_rho(stage_embs, scores, test_ids, probe)
    pooled_rhos[stage_name] = prho
    print(f"  Pooled rho: {prho:.4f}")

print(f"\n{'='*60}")
print("SUMMARY — Pooled Spearman rho by stage:")
for stage, rho in pooled_rhos.items():
    print(f"  {stage:8s}: {rho:.4f}")

In [ ]:
# Per-protein deltas: where does each protein lose rho?
common_pids = sorted(set.intersection(*[set(r.keys()) for r in per_protein_rhos.values()]))
print(f"Proteins with rho at all stages: {len(common_pids)}")

deltas = {}
for pid in common_pids:
    raw_rho = per_protein_rhos["raw"][pid]
    abtt_rho = per_protein_rhos["abtt"][pid]
    rp_rho = per_protein_rhos["rp768"][pid]
    pq_rho = per_protein_rhos["pq"][pid]
    deltas[pid] = {
        "raw_rho": raw_rho, "final_rho": pq_rho,
        "delta_abtt": abtt_rho - raw_rho,
        "delta_rp": rp_rho - abtt_rho,
        "delta_pq": pq_rho - rp_rho,
        "delta_total": pq_rho - raw_rho,
    }

delta_arr = np.array([d["delta_total"] for d in deltas.values()])
abtt_drops = np.array([d["delta_abtt"] for d in deltas.values()])
rp_drops = np.array([d["delta_rp"] for d in deltas.values()])
pq_drops = np.array([d["delta_pq"] for d in deltas.values()])

print(f"\nTotal rho change (raw -> PQ decoded):")
print(f"  Mean: {np.mean(delta_arr):+.4f}, Median: {np.median(delta_arr):+.4f}")
print(f"  Worst: {np.min(delta_arr):+.4f}, Best: {np.max(delta_arr):+.4f}")
print(f"\nMean rho change per stage:")
print(f"  ABTT:  {np.mean(abtt_drops):+.4f} (std={np.std(abtt_drops):.4f})")
print(f"  RP:    {np.mean(rp_drops):+.4f} (std={np.std(rp_drops):.4f})")
print(f"  PQ:    {np.mean(pq_drops):+.4f} (std={np.std(pq_drops):.4f})")

In [ ]:
# Visualization: per-stage rho distributions
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
stage_names = ["raw", "abtt", "rp768", "pq"]
stage_labels = ["Raw 1024d", "+ ABTT3", "+ RP 768d", "+ PQ M=192"]

for ax, sname, slabel in zip(axes, stage_names, stage_labels):
    vals = [per_protein_rhos[sname][p] for p in common_pids]
    ax.hist(vals, bins=25, alpha=0.7, edgecolor="black")
    ax.axvline(np.mean(vals), color="red", linestyle="--", label=f"mean={np.mean(vals):.3f}")
    ax.axvline(np.median(vals), color="blue", linestyle=":", label=f"med={np.median(vals):.3f}")
    ax.set_title(slabel)
    ax.set_xlabel("Per-protein Spearman ρ")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Count")
plt.suptitle("CheZOD-117: Per-Protein Disorder ρ at Each Pipeline Stage", fontsize=13)
plt.tight_layout()
plt.savefig("results/disorder_forensics/chezod_per_stage_rho.png", dpi=150)
plt.show()

In [ ]:
# Visualization: stage-wise rho drops
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
drop_data = [
    (abtt_drops, "ABTT3", "tab:blue"),
    (rp_drops, "RP 768d", "tab:orange"),
    (pq_drops, "PQ M=192", "tab:green"),
]
for ax, (drops, label, color) in zip(axes, drop_data):
    ax.hist(drops, bins=25, alpha=0.7, color=color, edgecolor="black")
    ax.axvline(0, color="black", linewidth=0.5)
    ax.axvline(np.mean(drops), color="red", linestyle="--", label=f"mean={np.mean(drops):+.3f}")
    ax.set_title(f"Δρ from {label}")
    ax.set_xlabel("Per-protein ρ change")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Count")
plt.suptitle("CheZOD-117: Per-Protein ρ Change at Each Stage", fontsize=13)
plt.tight_layout()
plt.savefig("results/disorder_forensics/chezod_stage_drops.png", dpi=150)
plt.show()

## Layer 1b: TriZOD Per-Stage Analysis

In [ ]:
print("Loading TriZOD data...")
tz_embs, tz_scores, tz_train_ids, tz_test_ids = h.load_trizod_data()
print(f"  Train: {len(tz_train_ids)}, Test: {len(tz_test_ids)}")

# Fit corpus stats on TriZOD training set
tz_train_embs = {k: tz_embs[k] for k in tz_train_ids if k in tz_embs}
tz_corpus_stats = compute_corpus_stats(tz_train_embs, n_sample=50_000, n_pcs=5, seed=42)

# Build stages
tz_stages = {}
tz_stages["raw"] = tz_embs
tz_stages["abtt"] = h.apply_abtt(tz_embs, tz_corpus_stats, k=3)
tz_stages["rp768"] = h.apply_rp(tz_stages["abtt"], d_out=768, seed=42)

tz_codec = OneEmbeddingCodec(d_out=768, quantization="pq", pq_m=192)
tz_codec.fit(tz_train_embs)
tz_stages["pq"] = h.apply_pq(tz_stages["rp768"], tz_codec)

# Train probes and compute rhos
tz_per_protein_rhos = {}
tz_pooled_rhos = {}
for stage_name, stage_embs in tz_stages.items():
    print(f"  {stage_name}...", end=" ", flush=True)
    probe = h.train_ridge_probe(stage_embs, tz_scores, tz_train_ids)
    tz_per_protein_rhos[stage_name] = h.per_protein_rho(stage_embs, tz_scores, tz_test_ids, probe)
    tz_pooled_rhos[stage_name] = h.pooled_rho(stage_embs, tz_scores, tz_test_ids, probe)
    print(f"pooled_ρ={tz_pooled_rhos[stage_name]:.4f}")

In [ ]:
# Combined CheZOD + TriZOD summary
print(f"{'Stage':<10} {'CheZOD pooled ρ':>16} {'TriZOD pooled ρ':>16}")
print("-" * 44)
for stage in ["raw", "abtt", "rp768", "pq"]:
    print(f"{stage:<10} {pooled_rhos[stage]:>16.4f} {tz_pooled_rhos[stage]:>16.4f}")

print(f"\n{'Stage':<10} {'CheZOD Δρ':>16} {'TriZOD Δρ':>16}")
print("-" * 44)
prev_cz, prev_tz = pooled_rhos["raw"], tz_pooled_rhos["raw"]
for stage in ["abtt", "rp768", "pq"]:
    cz_delta = pooled_rhos[stage] - prev_cz
    tz_delta = tz_pooled_rhos[stage] - prev_tz
    print(f"{stage:<10} {cz_delta:>+16.4f} {tz_delta:>+16.4f}")
    prev_cz, prev_tz = pooled_rhos[stage], tz_pooled_rhos[stage]

## Layer 2: Stratified Protein Selection

In [ ]:
# CheZOD: rank by total rho drop, select worst/average/winners
sorted_by_drop = sorted(deltas.items(), key=lambda x: x[1]["delta_total"])
worst_3 = [pid for pid, _ in sorted_by_drop[:3]]
best_3 = [pid for pid, _ in sorted_by_drop[-3:]]

drops_list = [d["delta_total"] for d in deltas.values()]
median_drop = np.median(drops_list)
by_distance_to_median = sorted(deltas.items(), key=lambda x: abs(x[1]["delta_total"] - median_drop))
average_3 = [pid for pid, _ in by_distance_to_median[:3]]

selected_cz = {"worst": worst_3, "average": average_3, "winners": best_3}

print("CheZOD selected proteins:\n")
print(f"{'PID':<12} {'Cat':<8} {'Len':>4} {'%Dis':>5} {'Z_mean':>7} {'Z_std':>6} "
      f"{'raw_ρ':>6} {'fin_ρ':>6} {'Δ_tot':>7} {'Δ_ABTT':>7} {'Δ_RP':>7} {'Δ_PQ':>7}")
print("-" * 100)
for category, pids in selected_cz.items():
    for pid in pids:
        d = deltas[pid]
        props = h.characterize_protein(scores, pid)
        print(f"{pid:<12} {category:<8} {props['n_residues']:>4} "
              f"{props['frac_disordered']:>5.1%} {props['z_mean']:>7.1f} {props['z_std']:>6.1f} "
              f"{d['raw_rho']:>6.3f} {d['final_rho']:>6.3f} "
              f"{d['delta_total']:>+7.3f} {d['delta_abtt']:>+7.3f} "
              f"{d['delta_rp']:>+7.3f} {d['delta_pq']:>+7.3f}")
    print()

In [ ]:
# TriZOD: same stratification
tz_common = sorted(set.intersection(*[set(r.keys()) for r in tz_per_protein_rhos.values()]))
tz_deltas = {}
for pid in tz_common:
    tz_deltas[pid] = {
        "raw_rho": tz_per_protein_rhos["raw"][pid],
        "final_rho": tz_per_protein_rhos["pq"][pid],
        "delta_abtt": tz_per_protein_rhos["abtt"][pid] - tz_per_protein_rhos["raw"][pid],
        "delta_rp": tz_per_protein_rhos["rp768"][pid] - tz_per_protein_rhos["abtt"][pid],
        "delta_pq": tz_per_protein_rhos["pq"][pid] - tz_per_protein_rhos["rp768"][pid],
        "delta_total": tz_per_protein_rhos["pq"][pid] - tz_per_protein_rhos["raw"][pid],
    }

tz_sorted = sorted(tz_deltas.items(), key=lambda x: x[1]["delta_total"])
tz_worst_3 = [pid for pid, _ in tz_sorted[:3]]
tz_best_3 = [pid for pid, _ in tz_sorted[-3:]]
tz_drops_list = [d["delta_total"] for d in tz_deltas.values()]
tz_median_drop = np.median(tz_drops_list)
tz_by_median = sorted(tz_deltas.items(), key=lambda x: abs(x[1]["delta_total"] - tz_median_drop))
tz_average_3 = [pid for pid, _ in tz_by_median[:3]]

selected_tz = {"worst": tz_worst_3, "average": tz_average_3, "winners": tz_best_3}

print("TriZOD selected proteins:\n")
for category, pids in selected_tz.items():
    print(f"  {category.upper()}:")
    for pid in pids:
        d = tz_deltas[pid]
        print(f"    {pid}: raw_ρ={d['raw_rho']:.3f} -> final_ρ={d['final_rho']:.3f} "
              f"(Δ={d['delta_total']:+.3f})")
    print()

## Layer 3: Subspace Forensics

Which directions in embedding space carry disorder signal? Does ABTT or RP destroy them?

In [ ]:
# Find disorder-informative dimensions: per-dimension correlation with Z-scores
X_all, y_all = [], []
for pid in train_ids:
    emb = embs[pid][:512]
    lab = np.asarray(scores[pid], dtype=np.float64)[:512]
    n = min(len(emb), len(lab))
    for i in range(n):
        if not np.isnan(lab[i]):
            X_all.append(emb[i])
            y_all.append(lab[i])

X_all = np.stack(X_all).astype(np.float32)
y_all = np.array(y_all, dtype=np.float64)
print(f"Training residues: {len(y_all)}, dimensions: {X_all.shape[1]}")

dim_correlations = np.zeros(X_all.shape[1])
for d in range(X_all.shape[1]):
    rho, _ = spearmanr(X_all[:, d], y_all)
    dim_correlations[d] = rho if not np.isnan(rho) else 0.0

top_k = 50
top_dims = np.argsort(np.abs(dim_correlations))[::-1][:top_k]
print(f"\nTop {top_k} disorder-correlated dimensions:")
print(f"  Max |ρ|: {np.abs(dim_correlations[top_dims[0]]):.4f} (dim {top_dims[0]})")
print(f"  Min |ρ| in top-{top_k}: {np.abs(dim_correlations[top_dims[-1]]):.4f}")
print(f"  Mean |ρ| across all dims: {np.mean(np.abs(dim_correlations)):.4f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(1024), np.abs(dim_correlations), width=1, alpha=0.5)
ax.bar(top_dims, np.abs(dim_correlations[top_dims]), width=1, color="red", alpha=0.8)
ax.set_xlabel("Dimension index")
ax.set_ylabel("|Spearman ρ| with Z-score")
ax.set_title("Per-Dimension Disorder Correlation (red = top 50)")
plt.tight_layout()
plt.savefig("results/disorder_forensics/dimension_disorder_correlation.png", dpi=150)
plt.show()

In [ ]:
# ABTT overlap: do removed PCs carry disorder signal?
top3_pcs = corpus_stats["top_pcs"][:3]  # (3, 1024)
disorder_direction = dim_correlations / (np.linalg.norm(dim_correlations) + 1e-10)

print("ABTT PC overlap with disorder direction:")
print(f"{'PC':>4} {'cos_sim':>10} {'weight on top-50':>18} {'expl_var':>10}")
print("-" * 46)
for i in range(5):
    pc = corpus_stats["top_pcs"][i]
    cos_sim = np.dot(pc, disorder_direction) / (np.linalg.norm(pc) + 1e-10)
    proj_on_top_dims = np.sum(pc[top_dims] ** 2) / np.sum(pc ** 2)
    removed = "REMOVED" if i < 3 else "kept"
    print(f"PC{i+1:>2} {cos_sim:>+10.4f} {proj_on_top_dims:>17.1%} {corpus_stats['explained_variance'][i]:>10.2f}  ({removed})")

# Variance of disorder-informative dims before/after ABTT
raw_sample = np.concatenate([embs[pid][:512] for pid in test_ids[:20]])
abtt_sample = np.concatenate([stages["abtt"][pid][:512] for pid in test_ids[:20]])

var_raw = np.var(raw_sample[:, top_dims], axis=0).sum()
var_abtt = np.var(abtt_sample[:, top_dims], axis=0).sum()
print(f"\nVariance in top-50 disorder dims:")
print(f"  Raw:  {var_raw:.2f}")
print(f"  ABTT: {var_abtt:.2f} ({var_abtt/var_raw:.1%} retained)")

In [ ]:
# RP preservation: how much disorder subspace survives projection?
rng = np.random.RandomState(42)
R = rng.randn(1024, 768).astype(np.float32)
Q, _ = np.linalg.qr(R, mode="reduced")
proj_matrix = Q * np.sqrt(1024 / 768)

# Per-dimension preservation after RP
all_preservation = np.array([
    np.linalg.norm(np.eye(1024, dtype=np.float32)[d] @ proj_matrix) ** 2
    for d in range(1024)
])
disorder_preservation = all_preservation[top_dims]

print(f"RP preservation analysis:")
print(f"  Expected (JL): {768/1024:.4f}")
print(f"  All dims mean:      {np.mean(all_preservation):.4f}")
print(f"  Disorder dims mean: {np.mean(disorder_preservation):.4f}")
print(f"  Disorder dims min:  {np.min(disorder_preservation):.4f}")
print(f"  Disorder dims max:  {np.max(disorder_preservation):.4f}")
print(f"  Difference:         {np.mean(disorder_preservation) - np.mean(all_preservation):+.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_preservation, bins=50, alpha=0.5, label="All 1024 dims", density=True)
ax.hist(disorder_preservation, bins=20, alpha=0.7, color="red", label="Top-50 disorder dims", density=True)
ax.axvline(768/1024, color="black", linestyle="--", label=f"Expected={768/1024:.3f}")
ax.set_xlabel("RP preservation (||proj(e_d)||^2)")
ax.set_ylabel("Density")
ax.set_title("RP Preservation: Disorder vs All Dimensions")
ax.legend()
plt.tight_layout()
plt.savefig("results/disorder_forensics/rp_preservation.png", dpi=150)
plt.show()

## Layer 4: Residue-Level Error Maps

In [ ]:
# Collect per-protein predictions at raw and PQ stages for selected CheZOD proteins
all_selected_pids = [p for cat in selected_cz.values() for p in cat]
raw_preds = h.per_protein_predictions(stages["raw"], scores, all_selected_pids, probes["raw"])
pq_preds = h.per_protein_predictions(stages["pq"], scores, all_selected_pids, probes["pq"])

# Residue trace plots for selected proteins
all_selected = [(cat, pid) for cat, pids in selected_cz.items() for pid in pids]
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes_flat = axes.flatten()

for idx, (category, pid) in enumerate(all_selected):
    ax = axes_flat[idx]
    if pid not in raw_preds or pid not in pq_preds:
        ax.set_title(f"{pid} — no data")
        continue

    y_true = raw_preds[pid]["y_true"]
    y_raw = raw_preds[pid]["y_pred"]
    y_pq = pq_preds[pid]["y_pred"]
    x = np.arange(len(y_true))

    ax.plot(x, y_true, "k-", alpha=0.7, linewidth=0.8, label="True Z")
    ax.plot(x, y_raw, "b-", alpha=0.6, linewidth=0.8, label="Raw probe")
    ax.plot(x, y_pq, "r-", alpha=0.6, linewidth=0.8, label="Compressed probe")
    ax.axhline(8.0, color="gray", linestyle=":", linewidth=0.5, label="Z=8 threshold")

    # Highlight residues where compressed is much worse
    error_raw = np.abs(y_true - y_raw)
    error_pq = np.abs(y_true - y_pq)
    worse = (error_pq - error_raw) > 2.0
    if worse.any():
        ax.scatter(x[worse], y_true[worse], color="red", s=10, zorder=5, label="Codec loss")

    d = deltas[pid]
    props = h.characterize_protein(scores, pid)
    ax.set_title(f"{pid} [{category}] Δρ={d['delta_total']:+.3f} "
                 f"({props['n_residues']}res, {props['frac_disordered']:.0%}dis)", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
    if idx >= 6:
        ax.set_xlabel("Residue")
    if idx % 3 == 0:
        ax.set_ylabel("Z-score")

plt.suptitle("CheZOD: True Z vs Predicted (Raw=blue, Compressed=red)", fontsize=13)
plt.tight_layout()
plt.savefig("results/disorder_forensics/chezod_residue_traces.png", dpi=150)
plt.show()

In [ ]:
# Embedding distortion vs prediction error
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, category in zip(axes, ["worst", "average", "winners"]):
    for pid in selected_cz[category]:
        rp_emb = stages["rp768"][pid][:512]
        pq_emb = stages["pq"][pid][:512]
        n = min(len(rp_emb), len(pq_emb))
        distortion = np.linalg.norm(rp_emb[:n] - pq_emb[:n], axis=1)

        if pid in raw_preds and pid in pq_preds:
            y_true = raw_preds[pid]["y_true"]
            err_raw = np.abs(raw_preds[pid]["y_pred"] - y_true)
            err_pq = np.abs(pq_preds[pid]["y_pred"] - y_true)
            delta_err = err_pq - err_raw
            n_plot = min(len(distortion), len(delta_err))
            ax.scatter(distortion[:n_plot], delta_err[:n_plot], alpha=0.3, s=5, label=pid)

    ax.set_xlabel("Embedding L2 distortion (RP->PQ)")
    ax.set_ylabel("Prediction error increase")
    ax.set_title(f"{category.upper()} proteins")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.legend(fontsize=7)

plt.suptitle("Embedding Distortion vs Prediction Error", fontsize=13)
plt.tight_layout()
plt.savefig("results/disorder_forensics/distortion_vs_error.png", dpi=150)
plt.show()

## Layer 5: Targeted Recovery Experiments

In [ ]:
# ABTT k sweep: does removing fewer PCs help disorder?
print("ABTT k sweep: pooled Spearman rho on CheZOD")
print(f"{'k':>3} {'Pooled ρ':>10} {'Δ vs raw':>10}")
print("-" * 25)

raw_rho = pooled_rhos["raw"]
print(f"{'raw':>3} {raw_rho:>10.4f} {'baseline':>10}")

for k in [0, 1, 2, 3, 4, 5]:
    if k == 0:
        centered = {pid: center_embeddings(emb, corpus_stats["mean_vec"])
                    for pid, emb in embs.items()}
        stage_embs = h.apply_rp(centered, d_out=768, seed=42)
    else:
        abtted = h.apply_abtt(embs, corpus_stats, k=k)
        stage_embs = h.apply_rp(abtted, d_out=768, seed=42)

    probe = h.train_ridge_probe(stage_embs, scores, train_ids)
    prho = h.pooled_rho(stage_embs, scores, test_ids, probe)
    print(f"{k:>3} {prho:>10.4f} {prho - raw_rho:>+10.4f}")

In [ ]:
# d_out sweep: does keeping more dimensions help disorder?
print("d_out sweep: pooled Spearman rho on CheZOD (ABTT k=3)")
print(f"{'d_out':>6} {'Pooled ρ':>10} {'Δ vs raw':>10} {'Dims lost':>10}")
print("-" * 40)

for d_out in [1024, 896, 832, 768, 640, 512]:
    if d_out >= 1024:
        stage_embs = stages["abtt"]
    else:
        stage_embs = h.apply_rp(stages["abtt"], d_out=d_out, seed=42)

    probe = h.train_ridge_probe(stage_embs, scores, train_ids)
    prho = h.pooled_rho(stage_embs, scores, test_ids, probe)
    print(f"{d_out:>6} {prho:>10.4f} {prho - raw_rho:>+10.4f} {1024 - d_out:>10}")

In [ ]:
# fp16 at 768d (no PQ) to isolate PQ contribution
rp_fp16 = h.apply_fp16(stages["rp768"])
probe_fp16 = h.train_ridge_probe(rp_fp16, scores, train_ids)
rho_fp16 = h.pooled_rho(rp_fp16, scores, test_ids, probe_fp16)

print(f"Isolating PQ contribution at 768d:")
print(f"  RP 768d (float32): {pooled_rhos['rp768']:.4f}")
print(f"  RP 768d (fp16):    {rho_fp16:.4f}")
print(f"  RP 768d + PQ:      {pooled_rhos['pq']:.4f}")
print(f"  fp16-only drop:    {rho_fp16 - pooled_rhos['rp768']:+.4f}")
print(f"  PQ-only drop:      {pooled_rhos['pq'] - pooled_rhos['rp768']:+.4f}")

## Summary

In [ ]:
print("=" * 70)
print("DISORDER INFORMATION LOSS — INVESTIGATION SUMMARY")
print("=" * 70)

print(f"\n1. PER-STAGE DECOMPOSITION (CheZOD pooled ρ):")
for stage, rho in pooled_rhos.items():
    print(f"   {stage:8s}: {rho:.4f}")

print(f"\n   TriZOD pooled ρ:")
for stage, rho in tz_pooled_rhos.items():
    print(f"   {stage:8s}: {rho:.4f}")

print(f"\n2. MEAN PER-PROTEIN ρ CHANGE BY STAGE (CheZOD):")
print(f"   ABTT:  {np.mean(abtt_drops):+.4f}")
print(f"   RP:    {np.mean(rp_drops):+.4f}")
print(f"   PQ:    {np.mean(pq_drops):+.4f}")

print(f"\n3. SELECTED PROTEINS:")
for ds_name, sel in [("CheZOD", selected_cz), ("TriZOD", selected_tz)]:
    print(f"   {ds_name}:")
    for cat, pids in sel.items():
        print(f"     {cat}: {pids}")

print(f"\n4. SUBSPACE ANALYSIS:")
print(f"   (See Layer 3 outputs above)")

print(f"\n5. RECOVERY EXPERIMENTS:")
print(f"   (See Layer 5 outputs above)")

print(f"\n{'='*70}")
print("NEXT STEPS:")
print("  1. Extract ESM2-650M + ESM-C 650M embeddings for CheZOD + TriZOD")
print("  2. Convert findings into experiments/45_disorder_forensics.py")
print("  3. Run validation script on all 3 PLMs")
print(f"{'='*70}")